# Enhanced Federated Learning Pipeline (Codespaces Driver)

This Codespaces-ready notebook focuses on essentials to start and monitor FL training.
Core modules are imported from standalone Python files in this workspace.

## 1) Environment and Paths (Codespaces)
Set workspace-first paths and conservative dataset caps before running the pipeline.

In [2]:
import os
import random
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
# Reduce memory spikes from graph JIT/XLA.
tf.config.optimizer.set_jit(False)

WORKDIR = Path.cwd()

# Codespaces-first local paths.
FRAMES_CANDIDATES = [
    WORKDIR / "ffpp_frames",
    Path("/workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/ffpp_frames"),
]
FRAMES_DIR = next((p for p in FRAMES_CANDIDATES if p.exists()), FRAMES_CANDIDATES[0])

MODEL_CANDIDATES = [
    WORKDIR / "efficientnetb4_final.keras",
    Path("/workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/efficientnetb4_final.keras"),
    Path("efficientnetb4_final.keras"),
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if p.exists()), MODEL_CANDIDATES[0])

# Smaller eval sets to reduce RAM pressure.
MAX_VAL_SAMPLES = 256
MAX_TEST_SAMPLES = 128

print(f"Workspace: {WORKDIR}")
print(f"Frames dir selected: {FRAMES_DIR}")
print(f"Frames dir exists: {FRAMES_DIR.exists()}")
print(f"Model path selected: {MODEL_PATH}")
print(f"Model exists: {MODEL_PATH.exists()}")

Workspace: /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection
Frames dir selected: /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/ffpp_frames
Frames dir exists: True
Model path selected: /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/efficientnetb4_final.keras
Model exists: True


In [3]:
# 1.5) Optional: GitHub sync for module + dataset updates
# Re-run this cell to pull latest code/data and place .py files in WORKDIR.

import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = os.environ.get(
    "FL_REPO_URL",
    "https://github.com/AverageWeebo101/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection.git",
)
REPO_BRANCH = os.environ.get("FL_REPO_BRANCH", "main")
CLONE_DIR = WORKDIR / "module_repo"

# If assets are under a subfolder in that repo, set it here (e.g., "src/fl").
REPO_ASSET_SUBDIR = "."

MODULE_FILES = [
    "enhanced_client_selection.py",
    "update_validation.py",
    "knowledge_distillation.py",
    "client_reputation_ledger.py",
    "evaluation_metrics.py",
    "federated_learning_cycle.py",
    "tff_data_utils.py",
    "tff_learning_process.py",
    "tff_federated_cycle.py",
]

# Optional helper modules: no warning if missing in repo.
OPTIONAL_MODULE_FILES = [
    "flwr_adapter.py",
]

DATA_FOLDERS = [
    "ffpp_tfrecord_clients",
    "ffpp_frames",
]

def _run(cmd):
    print("$", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, check=True, text=True)

if not CLONE_DIR.exists():
    _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)])
else:
    _run(["git", "-C", str(CLONE_DIR), "fetch", "origin", REPO_BRANCH])
    _run(["git", "-C", str(CLONE_DIR), "checkout", REPO_BRANCH])
    _run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

src_root = CLONE_DIR / REPO_ASSET_SUBDIR
if not src_root.exists():
    raise FileNotFoundError(f"Asset folder not found: {src_root}")

# 1) Copy required module files directly into WORKDIR (not nested in CLONE_DIR).
copied_modules = []
missing_modules = []
for fname in MODULE_FILES:
    src = src_root / fname
    dst = WORKDIR / fname
    if src.exists():
        shutil.copy2(src, dst)
        copied_modules.append(fname)
    else:
        missing_modules.append(fname)

# Optional module copy (silent if not found).
copied_optional_modules = []
for fname in OPTIONAL_MODULE_FILES:
    src = src_root / fname
    dst = WORKDIR / fname
    if src.exists():
        shutil.copy2(src, dst)
        copied_optional_modules.append(fname)

# 2) Copy data folders into WORKDIR.
copied_folders = []
missing_folders = []
for dname in DATA_FOLDERS:
    src_dir = src_root / dname
    dst_dir = WORKDIR / dname
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        copied_folders.append(dname)
    else:
        missing_folders.append(dname)

print(f"Copied {len(copied_modules)} required module files into {WORKDIR}")
print(f"Copied {len(copied_optional_modules)} optional module files into {WORKDIR}")
print(f"Synced {len(copied_folders)} data folders into {WORKDIR}")
if missing_modules:
    print("Missing required module files in repo (not copied):", missing_modules)
if missing_folders:
    print("Missing data folders in repo (not copied):", missing_folders)
print("Done. Re-run module verification, then continue pipeline cells.")

$ git clone --branch main https://github.com/AverageWeebo101/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection.git /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/module_repo


Cloning into '/workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/module_repo'...
fetch-pack: unexpected disconnect while reading sideband packet


KeyboardInterrupt: 

In [4]:
# 1.6) Verify clone is on latest remote commit
import subprocess

def _git_out(cmd):
    return subprocess.check_output(cmd, text=True).strip()

# Always fetch first so origin/<branch> reflects the newest remote state.
_run(["git", "-C", str(CLONE_DIR), "fetch", "origin", REPO_BRANCH])

local_head = _git_out(["git", "-C", str(CLONE_DIR), "rev-parse", "HEAD"])

remote_head = _git_out(["git", "-C", str(CLONE_DIR), "rev-parse", f"origin/{REPO_BRANCH}"])

latest_commit_line = _git_out([
    "git", "-C", str(CLONE_DIR),
    "log", "origin/" + REPO_BRANCH, "-1",
    "--pretty=format:%H | %ad | %an | %s", "--date=iso"
])

print("Local HEAD: ", local_head)
print("Remote HEAD:", remote_head)
print("Latest remote commit:")
print(latest_commit_line)

if local_head == remote_head:
    print("OK: Clone is up to date with remote branch.")
else:
    print("NOT LATEST: Local clone is behind remote. Re-run Cell 4.")

$ git -C /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/module_repo fetch origin main


fatal: cannot change to '/workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/module_repo': No such file or directory


CalledProcessError: Command '['git', '-C', '/workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/module_repo', 'fetch', 'origin', 'main']' returned non-zero exit status 128.

## 2) Verify Required Modules
These files were extracted from the old notebook and are now imported directly.

In [5]:
required_modules = [
    "enhanced_client_selection.py",
    "update_validation.py",
    "knowledge_distillation.py",
    "client_reputation_ledger.py",
    "evaluation_metrics.py",
    "federated_learning_cycle.py",
]

missing = [m for m in required_modules if not Path(m).exists()]
if missing:
    raise FileNotFoundError(f"Missing module files: {missing}")

print("All required module files are present.")

All required module files are present.


## 3) Build Capped Datasets
Creates lightweight val/test/proxy/sup datasets from frame paths using generator-backed tf.data pipelines.

In [6]:
import glob

all_paths = sorted(glob.glob(str(FRAMES_DIR / "**/*.jpg"), recursive=True))
assert len(all_paths) > 0, f"No frames found in {FRAMES_DIR}"

rng = np.random.RandomState(SEED)
idx = rng.permutation(len(all_paths))
all_paths = [all_paths[i] for i in idx]

def _path_to_label(path: str) -> np.float32:
    return np.float32(1.0 if "fake" in path.lower() else 0.0)

all_labels = [_path_to_label(p) for p in all_paths]
n = len(all_paths)

n_val = min(max(1, int(n * 0.15)), MAX_VAL_SAMPLES)
n_test = min(max(1, int(n * 0.10)), MAX_TEST_SAMPLES)
n_proxy = max(1, int(n * 0.015))
n_sup = max(1, int(n * 0.02))

val_paths = all_paths[:n_val]
val_labels = all_labels[:n_val]
test_paths = all_paths[n_val:n_val + n_test]
test_labels = all_labels[n_val:n_val + n_test]
proxy_paths = all_paths[n_val + n_test:n_val + n_test + n_proxy]
sup_paths = all_paths[n_val + n_test + n_proxy:n_val + n_test + n_proxy + n_sup]
sup_labels = all_labels[n_val + n_test + n_proxy:n_val + n_test + n_proxy + n_sup]

# Remaining samples become federated client training data.
train_start = n_val + n_test + n_proxy + n_sup
train_paths = all_paths[train_start:]
train_labels = all_labels[train_start:]
assert len(train_paths) > 0, "No training samples left after split; reduce val/test/proxy/sup caps."

print(
    f"Datasets built: train={len(train_paths)}, val={len(val_paths)}, "
    f"test={len(test_paths)}, proxy={len(proxy_paths)}, sup={len(sup_paths)}"
)

MODEL_IMG_SIZE = (224, 224)
if MODEL_PATH.exists():
    try:
        _tmp_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
        MODEL_IMG_SIZE = tuple(_tmp_model.input_shape[1:3])
        del _tmp_model
    except Exception as e:
        print(f"Model load warning (using default 224x224): {e}")

def _load_image(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(tf.image.resize(img, MODEL_IMG_SIZE), tf.float32)
    return img, label

def _load_image_only(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(tf.image.resize(img, MODEL_IMG_SIZE), tf.float32)
    return img

def _ds_from_paths_labels(paths, labels):
    return tf.data.Dataset.from_generator(
        lambda: ((p, np.float32(y)) for p, y in zip(paths, labels)),
        output_signature=(
            tf.TensorSpec(shape=(), dtype=tf.string),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        ),
    )

def _ds_from_paths(paths):
    return tf.data.Dataset.from_generator(
        lambda: (p for p in paths),
        output_signature=tf.TensorSpec(shape=(), dtype=tf.string),
    )

train_ds = _ds_from_paths_labels(train_paths, train_labels).map(_load_image, num_parallel_calls=1)
val_ds = _ds_from_paths_labels(val_paths, val_labels).map(_load_image, num_parallel_calls=1)
test_ds = _ds_from_paths_labels(test_paths, test_labels).map(_load_image, num_parallel_calls=1)
proxy_ds = _ds_from_paths(proxy_paths).map(_load_image_only, num_parallel_calls=1)
sup_ds = _ds_from_paths_labels(sup_paths, sup_labels).map(_load_image, num_parallel_calls=1)

Datasets built: train=30179, val=256, test=128, proxy=475, sup=633


E0000 00:00:1773972031.421550   13316 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 4) Configure and Start FL Pipeline
Imports only the orchestrator and config objects needed to run training.

In [7]:
from federated_learning_cycle import (
    FLCycleConfig,
    FederatedLearningCycle,
)

config = FLCycleConfig(
    model_path=str(MODEL_PATH),
    reports_dir="reports",
    num_devices=60,
    clients_per_round=8,
    local_epochs=2,
    global_rounds=30,
    local_batch_size=16,
    enable_distillation=True,
    validator_max_eval_batches=1,
    eval_every=10,
)

cycle = FederatedLearningCycle(config)
_ = cycle.load_global_model()

# Build federated client partitions from path lists (works with generator-backed datasets).
train_size = len(train_paths)
if train_size == 0:
    raise ValueError("Training split is empty; re-run Cell 9 and verify frame paths.")

num_clients = int(config.num_devices)
if train_size < num_clients:
    print(f"Warning: train samples ({train_size}) < num clients ({num_clients}); some clients get 1 sample.")

indices = np.arange(train_size)
np.random.seed(SEED)
np.random.shuffle(indices)
splits = np.array_split(indices, num_clients)

client_data = {}
for i, split_idx in enumerate(splits):
    cid = f"client_{i}"
    if len(split_idx) == 0:
        # Keep every client non-empty to avoid downstream edge cases.
        split_idx = np.array([indices[i % train_size]])
    c_paths = [train_paths[j] for j in split_idx]
    c_labels = [train_labels[j] for j in split_idx]
    client_data[cid] = _ds_from_paths_labels(c_paths, c_labels).map(
        _load_image, num_parallel_calls=1
    )

_ = cycle.create_clients(client_data)
cycle.setup_components()

print(
    f"Cycle initialized with {len(cycle.clients)} clients | "
    f"clients/round={config.clients_per_round}, rounds={config.global_rounds}, "
    f"local_epochs={config.local_epochs}"
)

2026-03-20 02:00:40,314 | INFO     | Loading global model from /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/efficientnetb4_final.keras …
2026-03-20 02:00:43,289 | INFO     | Global model loaded — 20,394,336 params, input shape (None, 260, 260, 3)
I0000 00:00:1773972045.988520   13316 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
2026-03-20 02:01:30,013 | INFO     | Created 100 federated clients.
2026-03-20 02:01:30,014 | INFO     | All FL-cycle components initialised.


Cycle initialized with 100 clients and ready to run.


In [8]:
import gc
import json
from pathlib import Path

import tensorflow as tf
from client_reputation_ledger import ClientReputationLedger
from federated_learning_cycle import convert_to_tflite

CKPT_DIR = WORKDIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path(config.reports_dir)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CKPT = CKPT_DIR / "latest_checkpoint.json"
HISTORY_FILE = CKPT_DIR / "history.json"
LEDGER_FILE = CKPT_DIR / "reputation_ledger.json"

# Keep recent checkpoint files only to limit disk/page-cache pressure.
MAX_CHECKPOINT_FILES = 3

# Export flags are disabled by default to avoid end-of-run kernel crashes.
# Run export in a separate fresh phase/cell for best stability.
EXPORT_TFLITE_AT_END = False
EXPORT_QUANTISED_AT_END = False

def _prune_old_weight_checkpoints(keep_last_n: int = MAX_CHECKPOINT_FILES) -> None:
    weight_files = sorted(CKPT_DIR.glob("round_*.weights.h5"))
    if len(weight_files) <= keep_last_n:
        return
    for stale in weight_files[:-keep_last_n]:
        try:
            stale.unlink()
        except Exception as e:
            print(f"Checkpoint cleanup warning for {stale.name}: {e}")

def _save_checkpoint(round_idx: int) -> None:
    weights_file = CKPT_DIR / f"round_{round_idx:03d}.weights.h5"
    cycle.global_model.save_weights(str(weights_file))
    if cycle.reputation_ledger is not None:
        cycle.reputation_ledger.save(str(LEDGER_FILE))
    HISTORY_FILE.write_text(json.dumps(cycle.history), encoding="utf-8")
    meta = {
        "round": int(round_idx),
        "weights_file": weights_file.name,
        "history_file": HISTORY_FILE.name,
        "ledger_file": LEDGER_FILE.name,
    }
    LATEST_CKPT.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    _prune_old_weight_checkpoints()

def _restore_if_available() -> int:
    if not LATEST_CKPT.exists():
        return 0
    meta = json.loads(LATEST_CKPT.read_text(encoding="utf-8"))
    last_round = int(meta.get("round", 0))
    weights_file = CKPT_DIR / meta.get("weights_file", "")
    if weights_file.exists():
        cycle.global_model.load_weights(str(weights_file))
        if cycle.validator is not None:
            cycle.validator.global_model.set_weights(cycle.global_model.get_weights())
    if HISTORY_FILE.exists():
        cycle.history = json.loads(HISTORY_FILE.read_text(encoding="utf-8"))
    if LEDGER_FILE.exists():
        restored = ClientReputationLedger.load(str(LEDGER_FILE))
        cycle.reputation_ledger = restored
        cycle.basic_ledger = restored.as_basic_ledger()
        if cycle.selector is not None:
            cycle.selector.reputation_ledger = cycle.basic_ledger
        if cycle.validator is not None:
            cycle.validator.ledger = cycle.basic_ledger
    print(f"Resuming from checkpoint after round {last_round}.")
    return last_round + 1

def _safe_tflite_export(global_model, output_path: str, quantise: bool) -> None:
    # Clear stale TF execution state before heavy conversion.
    tf.keras.backend.clear_session()
    gc.collect()
    convert_to_tflite(global_model, output_path, quantise=quantise)
    gc.collect()

start_round = _restore_if_available()
all_reports = []

if start_round == 0:
    print("No checkpoint found. Starting fresh run.")
    baseline_report = cycle.evaluator.evaluate(
        test_data=test_ds.take(128),
        batch_size=config.local_batch_size,
        federated_round=0,
        extra_info={"stage": "baseline", "eval_mode": "lightweight"},
        full_metrics=False,
        run_latency=False,
        max_samples_for_periodic_auc=128,
    )
    cycle.evaluator.save_report(baseline_report, tag="round_000_baseline")
    all_reports.append(baseline_report)
    _save_checkpoint(0)
    start_round = 1

for t in range(start_round, config.global_rounds + 1):
    info = cycle.execute_round(
        current_round=t,
        server_val_data=val_ds,
        proxy_data=(None if not config.enable_distillation else proxy_ds),
        supervised_data=(None if not config.enable_distillation else sup_ds),
    )

    cycle.history["round"].append(t)
    cycle.history["global_accuracy"].append(info["global_accuracy"])
    cycle.history["selected_clients"].append(info["selected"])
    cycle.history["num_accepted"].append(info["num_accepted"])
    cycle.history["num_rejected"].append(info["num_rejected"])
    cycle.history["distillation_loss"].append(info["distillation_loss"])

    is_eval_round = (t % config.eval_every == 0) or (t == 1) or (t == config.global_rounds)
    if is_eval_round:
        is_final_round = t == config.global_rounds
        eval_mode = "full" if is_final_round else "lightweight"
        eval_data = test_ds if is_final_round else test_ds.take(128)
        report = cycle.evaluator.evaluate(
            test_data=eval_data,
            batch_size=config.local_batch_size,
            federated_round=t,
            latency_max_batches=(None if is_final_round else 2),
            extra_info={
                "accepted": info["num_accepted"],
                "rejected": info["num_rejected"],
                "distillation_loss": info["distillation_loss"],
                "eval_mode": eval_mode,
            },
            full_metrics=is_final_round,
            run_latency=is_final_round,
            max_samples_for_periodic_auc=128,
        )
        cycle.evaluator.save_report(report, tag=f"round_{t:03d}")
        all_reports.append(report)

    _save_checkpoint(t)
    gc.collect()

if len(cycle.history.get("round", [])) >= config.global_rounds:
    if len(all_reports) > 1:
        cycle.evaluator.save_comparison_report(all_reports)
    if cycle.reputation_ledger is not None:
        cycle.reputation_ledger.save(str(REPORTS_DIR / "reputation_ledger_final.json"))

    if EXPORT_TFLITE_AT_END:
        _safe_tflite_export(
            cycle.global_model,
            config.tflite_output_path,
            quantise=False,
        )

        if EXPORT_QUANTISED_AT_END:
            _safe_tflite_export(
                cycle.global_model,
                config.tflite_output_path.replace(".tflite", "_quantised.tflite"),
                quantise=True,
            )
    else:
        print(
            "Skipped TFLite export in training cell for stability. "
            "Run the export cell after this in a fresh kernel/session."
        )

    cycle._print_summary()
else:
    print("Training paused before completion; resume by running this cell again.")

history = cycle.history
print("Training complete. History keys:", list(history.keys()))

2026-03-20 02:02:21,734 | INFO     | Starting evaluation for 'effnet_global' ...


No checkpoint found. Starting fresh run.


W0000 00:00:1773972142.358447   13316 cpu_allocator_impl.cc:82] Allocation of 155750400 exceeds 10% of free system memory.
W0000 00:00:1773972142.411703   13316 cpu_allocator_impl.cc:82] Allocation of 155750400 exceeds 10% of free system memory.
W0000 00:00:1773972142.469194   13316 cpu_allocator_impl.cc:82] Allocation of 155750400 exceeds 10% of free system memory.
W0000 00:00:1773972142.544202   13316 cpu_allocator_impl.cc:82] Allocation of 155750400 exceeds 10% of free system memory.
W0000 00:00:1773972142.575892   13316 cpu_allocator_impl.cc:82] Allocation of 155750400 exceeds 10% of free system memory.
2026-03-20 02:02:47,809 | INFO     | Classification - Acc: 0.8047 | F1-macro: 0.4459 | ROC-AUC: 0.0000
2026-03-20 02:02:47,909 | INFO     | Model size - params: 20,394,336 | disk: 77.80 MB
2026-03-20 02:02:47,911 | INFO     | Reports saved to effnet_global_20260320_020247_round_000_baseline.json and effnet_global_20260320_020247_round_000_baseline.txt
2026-03-20 02:02:49,015 | INFO 

: 

In [ ]:
import gc
import tensorflow as tf
from federated_learning_cycle import convert_to_tflite

EXPORT_STANDARD = True
EXPORT_QUANTISED = False

if not (EXPORT_STANDARD or EXPORT_QUANTISED):
    print("No exports selected. Set EXPORT_STANDARD and/or EXPORT_QUANTISED to True.")
else:
    # Ensure model object is present before conversion.
    if "cycle" not in globals() or cycle.global_model is None:
        raise RuntimeError(
            "Global model is not loaded. Run training setup/restore cells first, then retry export."
        )

    tf.keras.backend.clear_session()
    gc.collect()

    if EXPORT_STANDARD:
        convert_to_tflite(
            cycle.global_model,
            config.tflite_output_path,
            quantise=False,
        )
        gc.collect()

    if EXPORT_QUANTISED:
        convert_to_tflite(
            cycle.global_model,
            config.tflite_output_path.replace(".tflite", "_quantised.tflite"),
            quantise=True,
        )
        gc.collect()

    print("Export step complete.")

## 5) Export TFLite Notes
Use Cell 13 (the export code cell above) after training completes. For best stability, restart the kernel and restore the checkpoint before running export.

In [ ]:
# Optional cleanup: delete reports and checkpoints.
#import shutil
#from pathlib import Path

#workdir = globals().get("WORKDIR", Path.cwd())

#for p in [workdir / "reports", workdir / "checkpoints"]:
#    if p.exists():
        shutil.rmtree(p)
        print(f"Deleted: {p}")
#    else:
#        print(f"Not found: {p}")

Not found: /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/reports
Not found: /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/checkpoints
